# DDPG Training — Khí hậu Hà Nội (Colab GPU)

Theo **Guo et al., Applied Energy 2025** ([DOI](https://doi.org/10.1016/j.apenergy.2024.124467))

| Tham số | Giá trị |
|---------|--------|
| Khí hậu | Hà Nội (tháng 5–9) |
| Occupancy | **1 người cố định** |
| Pretrain | Fine-tune từ `checkpoints_v2` (Seoul) |
| Output | `checkpoints_hanoi/` + `actor_weights.npz` |

> **Bật GPU:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU*

In [ ]:
# 1. Clone repo & cài đặt
import os, sys

REPO = "https://github.com/trandat09062003/Smart_HVAC_AIOT.git"
ROOT = "Smart_HVAC_AIOT"

if not os.path.isdir(ROOT):
    !git clone {REPO}

os.chdir(f"{ROOT}/paper_reference")
sys.path.insert(0, os.getcwd())
print("CWD:", os.getcwd())

!pip install -q tensorflow==2.16 numpy matplotlib

In [ ]:
# 2. Kiểm tra GPU
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow {tf.__version__} | GPU: {len(gpus)}")
if not gpus:
    raise RuntimeError("Chưa bật GPU! Runtime → Change runtime type → T4 GPU")

In [ ]:
# 3. Cấu hình train (chỉnh tại đây)
os.environ["HANOI_EPISODES"] = "5000"       # paper: 5000
os.environ["HANOI_DAYS_PER_MONTH"] = "30"   # paper: 30 ngày/tháng

print("Episodes:", os.environ["HANOI_EPISODES"])
print("Days/month:", os.environ["HANOI_DAYS_PER_MONTH"])

In [ ]:
# 4. Train (resume checkpoints_hanoi nếu đã có)
!python train.py

In [ ]:
# 5. Đường cong huấn luyện
from IPython.display import Image, display

display(Image("logs/training_curve_hanoi.png"))

In [ ]:
# 6. Export actor → .npz (train.py đã export; cell này để chắc chắn)
import sys

sys.path.insert(0, "../server/mqtt-subscriber")
os.environ["CHECKPOINT_DIR"] = os.path.abspath("checkpoints_hanoi")

from load_model import export_actor_npz

out = os.path.abspath("../server/mqtt-subscriber/actor_weights.npz")
export_actor_npz(os.environ["CHECKPOINT_DIR"], out)
print("Exported:", out)

In [ ]:
# 7. Tải về máy
from google.colab import files

files.download("../server/mqtt-subscriber/actor_weights.npz")
files.download("checkpoints_hanoi/actor.weights.h5")
files.download("logs/training_curve_hanoi.png")

## Sau khi tải về

1. Copy `actor_weights.npz` → `server/mqtt-subscriber/`
2. Rebuild container trên server:

```bash
docker compose -p smart_hvac -f docker-compose.alt.yml build mqtt-subscriber
docker compose -p smart_hvac -f docker-compose.alt.yml up -d --no-deps mqtt-subscriber
```

3. Kiểm tra log: `docker logs ai-hvac-mqtt-subscriber --tail 20` → `Loaded DRL actor weights successfully!`